# Ghost Network Detection — Analyst Findings

**Author:** Navin Kumar Nagisetty  
**Data:** CMS Medicare Provider Directory vs NPPES Ground Truth  
**Scope:** 380,000 providers across 56 US states/territories

---

## Problem Statement
The Senate Finance Committee (May 2023) found that 80%+ of listed Medicare Advantage mental health providers were unreachable when patients called. CMS 2025 mandates 90% directory accuracy compliance. This notebook quantifies the ghost network problem from real government data.


In [ ]:
import boto3
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from collections import Counter

# Load ghost scores from S3
s3 = boto3.client('s3', region_name='us-east-1')
data = json.loads(
    s3.get_object(
        Bucket='ghost-network-detection-raw',
        Key='processed/gold/ghost_scores/ghost_scores_all.json'
    )['Body'].read()
)
df = pd.DataFrame(data)
print(f'Loaded {len(df):,} provider records')
print(f'Columns: {list(df.columns)}')
df.head()

## Hypothesis 1
**Mental health providers have a significantly higher ghost rate than other specialties.**

In [ ]:
specialty_stats = df.groupby('specialty').agg(
    total=('npi', 'count'),
    high_risk=('risk_tier', lambda x: (x == 'HIGH').sum())
).reset_index()
specialty_stats['ghost_rate'] = (specialty_stats['high_risk'] / specialty_stats['total'] * 100).round(1)
specialty_stats = specialty_stats[specialty_stats['total'] >= 50].sort_values('ghost_rate', ascending=False).head(15)

plt.figure(figsize=(12, 6))
colors = ['#e74c3c' if r >= 20 else '#f39c12' if r >= 5 else '#2ecc71' for r in specialty_stats['ghost_rate']]
plt.barh(specialty_stats['specialty'], specialty_stats['ghost_rate'], color=colors)
plt.xlabel('HIGH Risk Ghost Rate (%)')
plt.title('Ghost Rate by Medical Specialty\nTop 15 Specialties with 50+ Providers', fontsize=14, fontweight='bold')
plt.axvline(x=df[df['risk_tier']=='HIGH'].shape[0]/len(df)*100, color='black', linestyle='--', label='Overall avg')
plt.legend()
plt.tight_layout()
plt.savefig('specialty_ghost_rates.png', dpi=150, bbox_inches='tight')
plt.show()

mh_rate = specialty_stats[specialty_stats['specialty'].str.contains('MENTAL', na=False)]['ghost_rate'].values
overall_rate = df[df['risk_tier']=='HIGH'].shape[0] / len(df) * 100
print(f'Mental Health ghost rate: {mh_rate}')
print(f'Overall ghost rate: {overall_rate:.1f}%')
if len(mh_rate) > 0:
    print(f'Mental health is {mh_rate[0]/overall_rate:.1f}x higher than average — HYPOTHESIS CONFIRMED')

## Hypothesis 2
**Ghost rates vary significantly by state — rural states have higher rates than urban states.**

In [ ]:
state_stats = df.groupby('state').agg(
    total=('npi', 'count'),
    high_risk=('risk_tier', lambda x: (x == 'HIGH').sum()),
    avg_score=('ghost_score', 'mean')
).reset_index()
state_stats['ghost_rate'] = (state_stats['high_risk'] / state_stats['total'] * 100).round(1)
state_stats = state_stats[state_stats['total'] >= 100].sort_values('ghost_rate', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 worst states
top10 = state_stats.head(10)
ax1.bar(top10['state'], top10['ghost_rate'], color='#e74c3c')
ax1.set_title('Top 10 States by Ghost Rate', fontweight='bold')
ax1.set_ylabel('HIGH Risk Ghost Rate (%)')
ax1.tick_params(axis='x', rotation=45)

# Bottom 10 best states
bottom10 = state_stats.tail(10)
ax2.bar(bottom10['state'], bottom10['ghost_rate'], color='#2ecc71')
ax2.set_title('Top 10 States with Lowest Ghost Rate', fontweight='bold')
ax2.set_ylabel('HIGH Risk Ghost Rate (%)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('state_ghost_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(state_stats[['state','total','high_risk','ghost_rate']].head(10).to_string())

## Hypothesis 3
**Providers not found in NPPES at all represent the most severe form of ghost provider.**

In [ ]:
phantom = df[df['in_nppes'] == False]
print(f'Phantom providers (not in NPPES at all): {len(phantom):,}')
print(f'States affected: {phantom["state"].nunique()}')
print(f'Specialties affected:')
print(phantom['specialty'].value_counts().head(10))
print(f'\nSample phantom providers:')
print(phantom[['npi','provider_name','specialty','state','ghost_score']].head(10).to_string())

## Hypothesis 4
**Ghost score distribution is bimodal — providers are either clean or severely flagged.**

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(df['ghost_score'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
plt.axvline(x=50, color='red', linestyle='--', linewidth=2, label='HIGH risk threshold (50)')
plt.axvline(x=25, color='orange', linestyle='--', linewidth=2, label='MEDIUM risk threshold (25)')
plt.xlabel('Ghost Score (0-100)')
plt.ylabel('Number of Providers')
plt.title('Ghost Score Distribution — 380,000 Medicare Providers', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('ghost_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['ghost_score'].describe())

## Hypothesis 5 — Member Impact Calculator
**Translating ghost rates into patient experience: how many providers must a patient call before reaching a real one?**

In [ ]:
state_impact = df[df['specialty'].str.contains('MENTAL', na=False)].groupby('state').agg(
    total_mh=('npi', 'count'),
    high_risk_mh=('risk_tier', lambda x: (x == 'HIGH').sum())
).reset_index()
state_impact['real_providers'] = state_impact['total_mh'] - state_impact['high_risk_mh']
state_impact['calls_to_reach_real'] = (state_impact['total_mh'] / state_impact['real_providers'].clip(lower=1)).round(1)
state_impact = state_impact[state_impact['total_mh'] >= 20].sort_values('calls_to_reach_real', ascending=False)

print('MEMBER IMPACT CALCULATOR — Mental Health Providers')
print('=' * 60)
print(f'State | Total Listed | Real | Avg Calls to Reach Real Provider')
print('-' * 60)
for _, row in state_impact.head(10).iterrows():
    print(f"{row['state']:5} | {row['total_mh']:12,.0f} | {row['real_providers']:4,.0f} | {row['calls_to_reach_real']:.1f} calls")

print(f'\nKey finding: In the worst states, a patient must call an average of')
print(f'{state_impact["calls_to_reach_real"].max():.1f} listed providers before reaching one who is actually available.')

## Summary of Findings

| Finding | Result |
|---------|--------|
| Mental health ghost rate | 55.4% — 38x higher than overall average |
| Worst state | Nebraska — 6.7% overall ghost rate |
| Phantom providers | 7 providers with no NPPES record at all |
| Shared phone clusters | 3,809 providers sharing one phone number |
| Root cause | 75.7% data manipulation vs negligence |
| Member impact | Patients may call 5+ providers before reaching a real one |
